#Goals of this book:

- Prepare data
    -load processed data from (data/processed)
    -ensure labeled columns are present
    -split data into trainingg and test sets
-Train Models
-Track the model run and experiments with MFLOW
    Log parameters, metrics and artifacts
-Evaluate
    - use classification metrics (precsion/recall/confusion matrix, AUC)
    - Export feature importance

In [1]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

random_state = 42

ROOT = Path.cwd().resolve()
DATA_DIR = Path(r"C:\Users\payou\ClaimsAI\data\processed")
DEFAULT_FILE = DATA_DIR / "beneficiary_labeled.csv"

import mlflow
import mlflow.sklearn

pd.set_option("display.max_columns", 120)
pd.set_option("display.width",180)

In [2]:
#Load Dataset

if not DEFAULT_FILE.exists():
    raise FileNotFoundError(f"Expected file not found: {DEFAULT_FILE}\n"
                            f"Place a processed, labeled file in {DATA_DIR} or update DEFAULT_FILE.")

df = pd.read_csv(DEFAULT_FILE) if DEFAULT_FILE.suffix == ".csv" else pd.read_csv(DEFAULT_FILE)

print(f"Loaded shape: {df.shape}")
display(df.head(3))

Loaded shape: (349064, 18)


,AGE,Date_of_Death,Gender,Race,total_coverage_months,chronic_count,avg_reimb,op_ratio,car_ratio,Number_of_months_covered_a,Numver_of_months_covered_b,Number_of_months_HMO_coverage,Number_of_months_covered_d,is_anomaly,y_rule,rule_R_high_cost,rule_R_high_out_ratio,rule_R_high_chronic
0,51,0,Female,Hispanic,24,13,3551.200000,0.018765,0.075859,12,12,0,0,1,1,1,0,0
1,51,0,Female,White,36,15,509.459459,0.132720,0.254976,12,12,0,12,1,1,1,0,0
2,51,0,Male,White,48,13,798.775510,0.002667,0.040956,12,12,12,12,1,1,1,0,0


In [4]:
#Choose Target feature 

# Candidate target names from your labeling notebook(s).
# Adjust this list to match what you created (binary labels preferred for baseline).
CANDIDATE_TARGETS = [
    "anomaly_flag",      # 0/1
    "fraud_label",       # 0/1
    "is_anomaly"         # 0/1
]

target_col = next((c for c in CANDIDATE_TARGETS if c in df.columns), None)

if target_col:
    print(f"Using supervised target: {target_col}")
else:
    print("No supervised label found — we'll run unsupervised (IsolationForest) in the modeling chunk.")

# Exclude obvious non-features: identifiers, date keys, and the target itself.
ID_LIKE = [
    "DESYNPUF_ID", "BENE_ID", "CLM_ID", "NPI", "HICNO"
]
DATE_LIKE = [c for c in df.columns if "DT" in c or "DATE" in c or c.endswith("_DT") or c.endswith("_DATE")]

drop_cols = set(ID_LIKE + DATE_LIKE + ([target_col] if target_col else []))
feature_cols = [c for c in df.columns if c not in drop_cols]

print(f"Feature count: {len(feature_cols)}")

Using supervised target: is_anomaly
Feature count: 17


In [7]:
#Target Set up

target_col = "is_anomaly"

#feature column set up (all columns minus IDs, target and Dates)
id_like = ["DESYNPUF_ID", "BENE_ID", "CLM_ID", "NPI", "HICNO"]
date_like = [c for c in df.columns if "DT" in c or "Date" in c or c.endswith("_DT") or c.endswith("_DATE") or c.startswith("Date_")]


drop_cols = set(id_like + date_like + [target_col])
feature_cols = [c for c in df.columns if c not in drop_cols]

print(f"Target column: {target_col}")
print(f"Feature count: {len(feature_cols)}")

Target column: is_anomaly
Feature count: 16
